In [5]:
# The % ensures it installs in the correct Jupyter environment
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import glob
import subprocess
import pandas as pd
import json

import sys
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))


dani_dir = os.path.join(BASE_DIR, "src", "DANI")
run_dir = os.path.join(BASE_DIR, "src", "DANI", "training-runs", "00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered")
dataset_path = os.path.join(BASE_DIR, "src", "FakeCLR", "data", "panda.zip")
python_exec = os.path.join(BASE_DIR, "src", "DANI", "dani_env", "Scripts", "python.exe")

metrics_to_calc = "kid50k_full,fid50k_full,is50k"

In [8]:
import os
import glob
import subprocess

search_path = os.path.join(run_dir, "network-snapshot-*.pkl")
all_pkl_files = sorted(glob.glob(search_path))

if len(all_pkl_files) == 0:
    print(f"ERROR: No files found in {search_path}")
else:
    print(f"Found {len(all_pkl_files)} snapshots to evaluate.")
    print("Streaming live output...\n")

    # Force StyleGAN to use slow standard PyTorch ops instead of custom C++ plugins
    # This prevents the MSVC/GCC compiler error inside Jupyter.
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    
    # Adding an environment variable to disable the custom plugins.
    # Depending on the exact StyleGAN fork, one of these usually works:
    env["PYTORCH_NVFUSER_DISABLE"] = "1" 

    for pkl in all_pkl_files:
        print("\n" + "="*60)
        print(f"Evaluating: {os.path.basename(pkl)}")
        print("="*60)
        
        cmd = [
            python_exec,
            "calc_metrics.py",
            f"--metrics={metrics_to_calc}",
            f"--network={pkl}",
            f"--data={dataset_path}"
        ]
        
        # Notice we pass the modified `env` here
        process = subprocess.Popen(cmd, cwd=dani_dir, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        
        for line in process.stdout:
            print(line, end='') 
            
        process.wait()
        
        if process.returncode != 0:
            print(f"\n[!] Error evaluating {os.path.basename(pkl)}.")
        else:
            print(f"\n[*] Success.")
            
    print("\nAll snapshots evaluated!")

Found 13 snapshots to evaluate.
Streaming live output...


Evaluating: network-snapshot-000000.pkl
C:\Users\vishw\OneDrive\Desktop\Explo\DANI\torch_utils\ops\conv2d_gradfix.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
Loading network from "C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered\network-snapshot-000000.pkl"...
Dataset options:
{
  "class_name": "training.dataset.ImageFolderDataset",
  "path": "C:\\Users\\vishw\\OneDrive\\Desktop\\Explo\\FakeCLR\\data\\panda.zip",
  "resolution": 256,
  "use_labels": false
}
Launching processes...
Setting up PyTorch plugin "bias_act_plugin"... Failed!
Traceback (most recent call last):
  Fil

KeyboardInterrupt: 

In [ ]:
jsonl_files = sorted(glob.glob(os.path.join(run_dir, "metric-*.jsonl")))
all_metrics = []

for j_file in jsonl_files:
    with open(j_file, 'r') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    all_metrics.append(data)
                except json.JSONDecodeError:
                    continue

if all_metrics:
    df = pd.DataFrame(all_metrics)
    
    if 'snapshot_kimg' in df.columns:
        df = df.sort_values(by='snapshot_kimg')
        
    output_file = os.path.join(run_dir, "compiled_metrics.csv")
    df.to_csv(output_file, index=False)
    
    print(f"Saved to: {output_file}\n")
    display(df)
else:
    print("No metrics found.")

In [ ]:
# # ==========================================
# # 3. Aggregate JSONL files into a CSV
# # ==========================================
# print("Aggregating metrics into a single dataset...\n")

# jsonl_files = sorted(glob.glob(os.path.join(run_dir, "metric-*.jsonl")))
# all_metrics = []

# for j_file in jsonl_files:
#     with open(j_file, 'r') as f:
#         for line in f:
#             if line.strip():
#                 try:
#                     data = json.loads(line)
#                     all_metrics.append(data)
#                 except json.JSONDecodeError:
#                     continue

# if all_metrics:
#     # Convert to Pandas DataFrame
#     df = pd.DataFrame(all_metrics)
    
#     # Sort chronologically by the training tick
#     if 'snapshot_kimg' in df.columns:
#         df = df.sort_values(by='snapshot_kimg')
        
#     # Save for external use
#     output_file = os.path.join(run_dir, "compiled_metrics_over_time.csv")
#     df.to_csv(output_file, index=False)
    
#     print(f"Metrics successfully compiled and saved to: {output_file}\n")
    
#     # Display the dataframe natively in the notebook
#     display(df)
# else:
#     print("No metrics found to aggregate. Double-check the console output from Cell 3.")